In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore") # umap deprecation warnings related to numba do not concern us.


In [ ]:
from pathlib import Path as path
import functools as fn

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scboolseq.utils import parse_data_directory, parse_pickles
from scboolseq import scBoolSeq

from markers import (
    positive_markers, # function to generate a configuration dictionnary from a set of marker genes
    states_and_markers, # Dictionnary containing phenotypes and their marker genes
    marker_genes, # pandas.Series containing simply the name of markers
)


### TF database loading

In [1]:
import decoupler as dc


In [3]:
import os

# Check if "collectri_mouse.csv" exists; if not, download it
collectri_file = "collectri_mouse.csv"
if os.path.exists(collectri_file):
    net = pd.read_csv(collectri_file)
else:
    net = dc.op.collectri(organism="mouse", remove_complexes=False)
    net.to_csv(collectri_file, index=False)


In [5]:
net[net["weight"] != 1.0]

,source,target,weight,resources,references,sign_decision
14,Trp53,Igf1r,-1.0,ExTRI;GEREDB;NTNU.Curated;TRRUST,10023442;11001824;12089352;12444079;15131018;1...,PMID
15,Zbtb16,Ccna2,-1.0,DoRothEA-A;ExTRI;TRRUST,10023668;11920165;14527952;18504436;20731660,PMID
16,Etv6,Il3,-1.0,ExTRI;NTNU.Curated,10023677,PMID
17,Runx1,Il3,-1.0,CytReg;DoRothEA-A;ExTRI;TRRUST,10023677;11997340;15735013;20506188;8180380;90...,PMID
34,Trp53,Rb1,-1.0,DoRothEA-A;ExTRI;GEREDB;HTRI;NTNU.Curated;TFac...,10029406;11121443;11244564;11818069;11850820;1...,PMID
...,...,...,...,...,...,...
42942,Etv1,Hes5,-1.0,Pavlidis2021,22735158,PMID
42950,Jazf1,Mef2c,-1.0,Pavlidis2021,26101156,PMID
42964,Wnt3a,Lef1,-1.0,Pavlidis2021,19650108,PMID
43070,Ctcf,Tnfrsf18,-1.0,DoRothEA-A,20219941;22951020,PMID


## Method selection

This notebook consolidates the three aggregation variants (`Sum`, `Mode`, `Bootstrap`).
Change **`AGGREGATION`** below; the rest of the pipeline reuses the same steps with method-specific defaults and file prefixes.

| `AGGREGATION` | Observation tags (default) | Aggregation | BoNesis PKN (default) | Extra constraint |
|---|---|---|---|---|
| `"sum"` | `pseudotime_inferred_observations.csv` | sum + `SymmetricBinarizer` | `pkn_biggest_wcc` | — |
| `"mode"` | `pseudotime_inferred_observations.csv` | pandas `.mode()` | `pkn_biggest_wcc` | — |
| `"bootstrap"` | `scboolseq_inferred_observations.csv` | bootstrap mean + CI | `prior_grn` | `HC` |

You can override individual options after selecting the method.


In [ ]:
# Choose one of: "sum", "mode", "bootstrap"
AGGREGATION = "sum"

assert AGGREGATION in {"sum", "mode", "bootstrap"}, AGGREGATION

# Defaults matching the original three notebooks
_DEFAULTS = {
    "sum": {
        "obs_tags_file": "pseudotime_inferred_observations.csv",
        "pkn_source": "biggest_wcc",   # vs "prior_grn"
        "include_hc_constraint": False,
        "export_aggregated_csv": True,
    },
    "mode": {
        "obs_tags_file": "pseudotime_inferred_observations.csv",
        "pkn_source": "biggest_wcc",
        "include_hc_constraint": False,
        "export_aggregated_csv": False,
    },
    "bootstrap": {
        "obs_tags_file": "scboolseq_inferred_observations.csv",
        "pkn_source": "prior_grn",
        "include_hc_constraint": True,
        "export_aggregated_csv": False,
    },
}

OBS_TAGS_FILE = _DEFAULTS[AGGREGATION]["obs_tags_file"]
PKN_SOURCE = _DEFAULTS[AGGREGATION]["pkn_source"]  # "biggest_wcc" | "prior_grn"
INCLUDE_HC_CONSTRAINT = _DEFAULTS[AGGREGATION]["include_hc_constraint"]
EXPORT_AGGREGATED_CSV = _DEFAULTS[AGGREGATION]["export_aggregated_csv"]
FILE_PREFIX = AGGREGATION  # used for output filenames

print(f"AGGREGATION={AGGREGATION!r}")
print(f"OBS_TAGS_FILE={OBS_TAGS_FILE!r}")
print(f"PKN_SOURCE={PKN_SOURCE!r}")
print(f"INCLUDE_HC_CONSTRAINT={INCLUDE_HC_CONSTRAINT}")
print(f"FILE_PREFIX={FILE_PREFIX!r}")


In [ ]:
%ls *csv


In [ ]:
data = pd.read_csv("GSE122466_Retina_vargenes_batch1.csv", index_col=0)
print(data.shape)
data.head()


In [ ]:
obs_tags = pd.read_csv(OBS_TAGS_FILE, index_col=0)
obs_tags.observation.value_counts()


In [ ]:
%%time
scbool = scBoolSeq(
    dor_threshold=0.995, # To retain one of the markers
    confidence=.75 # To maximize binarization
)
scbool.fit(data)


In [ ]:
scbool.criteria_.Category.value_counts()


In [ ]:
scbool.criteria_.loc[marker_genes, :].Category.value_counts()


In [ ]:
%time bin_data = scbool.binarize(data)


In [ ]:
partial_bin_configs = bin_data[marker_genes]
partial_bin_configs.fillna('').head()


In [ ]:
binarized_groups = {}
for obs, frame in obs_tags.groupby('observation'):
    binarized_groups.update({
        obs: bin_data.loc[frame.index, :]
    })


In [ ]:
binarized_groups.keys()


### Aggregate groups

Depending on `AGGREGATION`:

**sum** — encode False/\?/True as −1/0/1, sum within each phenotype, then apply `SymmetricBinarizer`:

$
\begin{equation}
    f(x) =
    \begin{cases}
        -1 &       \text{if}\;\; x = False\\
         0 &       \text{if}\;\;  x = \text{?}\\
         1 &       \text{if}\;\; x = True
    \end{cases}
\end{equation}
$

**mode** — take the first pandas `.mode()` per gene within each phenotype.

**bootstrap** — same −1/0/1 encoding, then bootstrap estimation of the mean; a gene is 1 / 0 / NaN if its CI is entirely positive / entirely negative / otherwise.


In [ ]:
%%time

if AGGREGATION == "sum":
    from scboolseq.binarization import SymmetricBinarizer

    _sym_thresh = .3
    agg_obs = []
    for group_name, binarized_group in binarized_groups.items():
        _mapped_meta = binarized_group.replace(0.0, -1.0).fillna(0.0)
        _meta_bin = (
            SymmetricBinarizer(threshold=_sym_thresh*_mapped_meta.shape[0], require_df=True)
                .fit_transform(
                    _mapped_meta.sum().to_frame()
                )
        )
        _meta_bin.columns = [group_name]
        agg_obs.append(_meta_bin)
    meta = pd.concat(agg_obs, axis=1).T

elif AGGREGATION == "mode":
    agg_obs = []
    for group_name, binarized_group in binarized_groups.items():
        _meta_bin = binarized_group.mode(dropna=False).loc[0, ].to_frame() # Only take the first mode, otherwise there can be multiple rows
        _meta_bin.columns = [group_name]
        agg_obs.append(_meta_bin)
    meta = pd.concat(agg_obs, axis=1).T

elif AGGREGATION == "bootstrap":
    from sklearn.utils import Bunch
    from scboolseq.meta.bootstrap import bootstrap

    def aggregate_bootstrap_frame(b_ci_frame: pd.DataFrame, name: str):
        positive_mask = (b_ci_frame > 0).all(axis=0)
        negative_mask = (b_ci_frame < 0).all(axis=0)
        aggregated = pd.Series(np.nan, index=b_ci_frame.columns, name=name)
        aggregated[positive_mask] = 1.0
        aggregated[negative_mask] = 0.0
        return aggregated

    mapped_binarized_groups = Bunch(**{
        group_name: binarized_group.replace(0.0, -1.0).fillna(0.0)
        for group_name, binarized_group
        in binarized_groups.items()
    })

    bootstrap_frames = {}
    meta_bin_ls = []
    for name, frame in mapped_binarized_groups.items():
        print(name)
        _b = frame.apply(bootstrap)
        bootstrap_frames.update({name: _b})
        meta_bin_ls.append(aggregate_bootstrap_frame(_b, name))

    meta = pd.concat(meta_bin_ls, axis=1).T

else:
    raise ValueError(AGGREGATION)

# Aliases kept for readability / compatibility with the original notebooks
meta_sum = meta_mode = meta_boot = meta_bootstrap_bin = meta
print(AGGREGATION, meta.shape)
meta


In [ ]:
for pheno, genes in states_and_markers.items():
    if pheno in meta.index:
        print(pheno, "\t:\t", meta.loc[pheno, list(genes)].to_dict())


In [ ]:
meta[marker_genes].T


In [ ]:
if EXPORT_AGGREGATED_CSV:
    _agg_csv = f"meta_{FILE_PREFIX}_bin_aggregated.csv"
    meta.to_csv(_agg_csv)
    print("Wrote", _agg_csv)


In [ ]:
null_var_mask = meta.var(skipna=False) == 0
null_var_genes = meta.columns[null_var_mask]
null_var_genes.shape


In [ ]:
with open(f"{FILE_PREFIX}_active_invariant_genes.txt", "w") as _active_genes, open(f"{FILE_PREFIX}_inactive_invariant_genes.txt", "w") as _inactive_genes:
    _active_genes.write(f"Gene\n")
    _inactive_genes.write(f"Gene\n")
    for _gene, _val in meta[null_var_genes].mode().T.itertuples(index=True):
        if _val:
            _active_genes.write(f"{_gene}\n")
        else:
            _inactive_genes.write(f"{_gene}\n")


In [ ]:
print(meta.shape)
meta_var = meta
meta.iloc[:, :10]


In [ ]:
dorothea_db = pd.read_csv("dorothea_mouse_tfs.csv")
dorothea_db.columns = dorothea_db.columns[:-1].to_list() + ['sign']
dorothea_db.head()


In [ ]:
def in_db(frame: pd.DataFrame, db: pd.DataFrame):
    is_tf = frame.columns.isin(db.tf)
    is_target = frame.columns.isin(db.target)
    in_db = pd.Series(
        fn.reduce(np.logical_or, [is_tf, is_target]),
        index=frame.columns
    )
    return in_db


In [ ]:
in_db(meta_var, dorothea_db).mean()


In [ ]:
in_db(meta_var[marker_genes], dorothea_db).mean()


In [ ]:
meta_var[null_var_genes].columns[~in_db(meta_var[null_var_genes], dorothea_db)]


In [ ]:
print(meta_var.shape)
meta_var_db = meta_var[meta_var.columns[in_db(meta_var, dorothea_db)]]
print(meta_var_db.shape)
meta_var_db.iloc[:, :10]


In [ ]:
meta_var_db.T.isna().mean()


In [ ]:
meta_var_db.isna().mean().mean()


In [ ]:
import networkx as nx
from grn import *


In [ ]:
dorothea_db.confidence.unique()


In [ ]:
# The whole grn is weakly connected if we consider all confidence levels
whole_grn = df_to_graph(dorothea_db.query("confidence != 'E' & confidence != 'D'"))
induced_grn = nx.induced_subgraph(whole_grn, meta_var_db.columns)
len(whole_grn), len(induced_grn)


In [ ]:
core_tf_nw = extract_largest_scc(whole_grn)
len(core_tf_nw)


In [ ]:
prior_grn = (
    nx.induced_subgraph(
        whole_grn,
        set(core_tf_nw).union(
            meta_var_db.columns
        )
    )
)
len(prior_grn)


In [ ]:
nx.is_weakly_connected(prior_grn)


In [ ]:
pkn_biggest_wcc = nx.subgraph(prior_grn, max(nx.weakly_connected_components(prior_grn), key=len))
len(pkn_biggest_wcc)


In [ ]:
all(i in pkn_biggest_wcc for i in core_tf_nw)


In [ ]:
marker_genes.isin(prior_grn).mean()


In [ ]:
marker_genes.isin(pkn_biggest_wcc).mean()


In [ ]:
print("\t".join(["Cell", "pTotal", "pBiggestWCC"]))
for pheno, genes in states_and_markers.items():
    if pheno in meta.index:
        print("\t".join(
            (lambda __: f"{100*__:.2f}" if isinstance(__, float) else __)(_)
            for _ in
            [pheno, pd.Index(genes).isin(whole_grn).mean(), pd.Index(genes).isin(pkn_biggest_wcc).mean()]
        ))


In [ ]:
import bonesis


In [ ]:
def df_to_bonesis_data(df: pd.DataFrame):
    data = {}
    for config, genes in df.iterrows():
        data.update({config: genes.dropna().to_dict()})
    return data


In [ ]:
data_meta = df_to_bonesis_data(meta_var_db)
data_meta.keys()


In [ ]:
if PKN_SOURCE == "prior_grn":
    _pkn_graph = prior_grn
elif PKN_SOURCE == "biggest_wcc":
    _pkn_graph = pkn_biggest_wcc
else:
    raise ValueError(PKN_SOURCE)

pkn = bonesis.domains.InfluenceGraph(
    _pkn_graph, maxclause=8, allow_skipping_nodes=True, canonic=False
)
print(f"PKN_SOURCE={PKN_SOURCE!r}, nodes={len(_pkn_graph)}")


In [ ]:
bo = bonesis.BoNesis(pkn, data_meta)
bo


In [ ]:
import json
with open(f"{FILE_PREFIX}_prior_grn.json", "w") as _prior_grn, open(f"{FILE_PREFIX}_data_meta.json", "w") as _data:
    json.dump(nx.node_link_data(pkn), _prior_grn)
    json.dump(data_meta, _data)


In [ ]:
~bo.obs("RPC") >= ~bo.obs("NB1") >= ~bo.obs("NB2")
~bo.obs("NB2") >= bo.fixed(~bo.obs("Cones"))
~bo.obs("NB2") >= bo.fixed(~bo.obs("RGC"))
~bo.obs("NB2") >= bo.fixed(~bo.obs("AC"))
if INCLUDE_HC_CONSTRAINT:
    ~bo.obs("NB2") >= bo.fixed(~bo.obs("HC"))
None


In [ ]:
import datetime
from pathlib import Path as path
from scboolseq.utils import Timer


In [ ]:
date = datetime.datetime.now()
bo.maximize_nodes()
bo.maximize_strong_constants()
view = bonesis.NonStrongConstantNodesView(bo, mode="optN")
view.standalone(
    output_filename=f"optim_dorothea_v2_{FILE_PREFIX}_{date.strftime('%Y-%m-%d')}.sh"
)
print("Standalone shell file written", flush=True)

kept_nodes = {}
with Timer("Optimizing NonStrongConstantNodesView(bo, mode='optN')"):
    for kept_nodes in view:
        break

_replica_grn = nx.subgraph(prior_grn, kept_nodes)


In [ ]:
len(kept_nodes)


In [ ]:
marker_genes.isin(_replica_grn).mean()


In [ ]:
bonesis.InfluenceGraph(_replica_grn)


In [ ]:
EXPORT = True

if EXPORT:
    nx.nx_pydot.write_dot(extract_largest_scc(bonesis.InfluenceGraph(_replica_grn)), f"{FILE_PREFIX}_grn_scc_v1.dot")
    import json
    with open(f"{FILE_PREFIX}_grn_optimized_v1.json", "w") as f:
        json.dump(nx.node_link_data(_replica_grn), f)

    with open(f"{FILE_PREFIX}_pkn_bonesis_obj_optimized_v1.json", "w") as f:
        json.dump(nx.node_link_data(bonesis.InfluenceGraph(_replica_grn)), f)
    print("Exported")
else:
    print("Not exported")
